# Orchestrator-Workers: el manager decide sobre la marcha

Clasificación: **Proceso jerárquico.** Un manager LLM lee la petición del cliente y decide qué agentes necesita, en qué orden y cuánto trabajo darle a cada uno.

A diferencia de parallelization (donde las 4 tasks están fijadas antes de arrancar), aquí el manager adapta la ejecución al caso concreto. Un cliente que solo quiere relajarse necesita más trabajo de actividades; uno con itinerario ajustado, más de vuelos.

## Cómo funciona en CrewAI

`Process.hierarchical` activa un manager automático que orquesta a los agentes. El manager recibe la task principal, decide a quién delegar, y puede volver a consultar al mismo agente si necesita más detalle.

```python
crew = Crew(
    agents=[...],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
)
```

In [1]:
!uv pip install -r requirements.txt --quiet

In [2]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

In [6]:
from crewai import Agent, Task, Crew, Process
from viajes_crew import ViajesCrew

base_crew = ViajesCrew()

peticion_cliente = (
    "Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. "
    "Lo unico que de verdad importa es ver auroras boreales y banarnos en fuentes termales; "
    "el resto (vuelos, alojamiento, alquiler de coches, rutas, transporte) lo he revisado ya manualmente."
)

manager = Agent(
    role="LLM Manager",
    goal="Decide which specialized agents should be invoked to deliver the user's request.",
    backstory="You are a manager agent in an Orchestrator-Worker setup. You're responsible for deciding the most suitable agents for a particular request",
    llm="gpt-4o-mini",
    verbose=True
)

main_task = Task(
    description=(
        f"Peticion del cliente: {peticion_cliente}\n\n"
        "Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. "
        "Entrega un itinerario completo y el presupuesto desglosado."
    ),
    expected_output="Itinerario dia a dia con vuelos, alojamiento, actividades y transporte, coste por partida y total dentro del presupuesto indicado.",
    agent=manager,
)

crew = Crew(
    agents=[base_crew.vuelos(), base_crew.alojamiento(), base_crew.actividades(), base_crew.transporte()],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
    verbose=True,
)

result = await crew.kickoff_async()
print(result.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.14                                                                                       │
│  Latest version:  1.15.16                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4c270f23-d02d-4d51-a5dd-1feb2536da4f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento, alquiler de     │
│  coches, rutas, transporte) lo he revisado ya manualmente.                                                      │
│                                                                                                                 │
│  Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. Entrega un itinerario     │
│  completo y el presupuesto desglosado.                                                                          │
│  ID: 74b8e0d9-082d-4e0b-8aa6-5d3d55bd6eb0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento, alquiler de     │
│  coches, rutas, transporte) lo he revisado ya manualmente.                                                      │
│                                                                                                                 │
│  Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. Entrega un itinerario     │
│  completo y el presupuesto desglosado.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#8) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Investigar y sugerir lugares e itinerarios para baños en fuentes termales en Islandia,         │
│  incluyendo horarios, precios y disponibilidad.', 'context': 'El cliente también enfatiza la importancia...     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Itinerario y Presupuesto para Viaje a Islandia (5 Días, 2 Personas)                                        │
│                                                                                                                 │
│  **Presupuesto Total: 2200 EUR (aproximadamente 324,000 ISK)**                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **Día 1: Llegada a Islandia**                                                                              │
│  - **Llegada**: Aeropuerto de Keflavik.                                                                         │
│  - **Actividad**: Visita a Blue Lagoon.                                                                         │
│    - **Costo**: 7,000 ISK por persona (14,000 ISK total).                                                       │
│  - **Alojamiento**: Hotel en Reikiavik.                                                                         │
│    - **Costo**: 30,000 ISK (recomendado en el centro, posibilidad de reservas a través de Booking).             │
│  - **Transporte**: Alquiler de coche.                                                                           │
│    - **Costo**: 13,000 ISK (diario).                                                                            │
│                                                                                                                 │
│  #### **Subtotal Día 1**: 57,000 ISK                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **Día 2: Círculo Dorado**                                                                                  │
│  - **Actividad**: Excursión al Círculo Dorado.                                                                  │
│    - Incluye: Parques Nacional Thingvellir, Geysir y Gullfoss.                                                  │
│  - **Fuentes Termales**: Parada en Secret Lagoon.                                                               │
│    - **Costo**: 3,000 ISK por persona (6,000 ISK total).                                                        │
│  - **Transporte**: Usar el vehículo alquilado.                                                                  │
│                                                                                                                 │
│  #### **Subtotal Día 2**: 39,000 ISK                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento, alquiler de     │
│  coches, rutas, transporte) lo he revisado ya manualmente.                                                      │
│                                                                                                                 │
│  Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. Entrega un itinerario     │
│  completo y el presupuesto desglosado.                                                                          │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Itinerario y Presupuesto para Viaje a Islandia (5 Días, 2 Personas)

**Presupuesto Total: 2200 EUR (aproximadamente 324,000 ISK)**

---

### **Día 1: Llegada a Islandia**
- **Llegada**: Aeropuerto de Keflavik.
- **Actividad**: Visita a Blue Lagoon.
  - **Costo**: 7,000 ISK por persona (14,000 ISK total).
- **Alojamiento**: Hotel en Reikiavik.
  - **Costo**: 30,000 ISK (recomendado en el centro, posibilidad de reservas a través de Booking).
- **Transporte**: Alquiler de coche.
  - **Costo**: 13,000 ISK (diario).
  
#### **Subtotal Día 1**: 57,000 ISK

---

### **Día 2: Círculo Dorado**
- **Actividad**: Excursión al Círculo Dorado.
  - Incluye: Parques Nacional Thingvellir, Geysir y Gullfoss.
- **Fuentes Termales**: Parada en Secret Lagoon.
  - **Costo**: 3,000 ISK por persona (6,000 ISK total).
- **Transporte**: Usar el vehículo alquilado.

#### **Subtotal Día 2**: 39,000 ISK

---

### **Día 3: Norte de Islandia**
- **Actividad**: Viaje a Myvatn Nature Baths.
  - **Costo**: 4,000 IS

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4c270f23-d02d-4d51-a5dd-1feb2536da4f                                                                       │
│  Final Output: ### Itinerario y Presupuesto para Viaje a Islandia (5 Días, 2 Personas)                          │
│                                                                                                                 │
│  **Presupuesto Total: 2200 EUR (aproximadamente 324,000 ISK)**                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **Día 1: Llegada a Islandia**                                                                              │
│  - **Llegada**: Aeropuerto de Keflavik.                                                                         │
│  - **Actividad**: Visita a Blue Lagoon.                                                                         │
│    - **Costo**: 7,000 ISK por persona (14,000 ISK total).                                                       │
│  - **Alojamiento**: Hotel en Reikiavik.                                                                         │
│    - **Costo**: 30,000 ISK (recomendado en el centro, posibilidad de reservas a través de Booking).             │
│  - **Transporte**: Alquiler de coche.                                                                           │
│    - **Costo**: 13,000 ISK (diario).                                                                            │
│                                                                                                                 │
│  #### **Subtotal Día 1**: 57,000 ISK                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **Día 2: Círculo Dorado**                                                                                  │
│  - **Actividad**: Excursión al Círculo Dorado.                                                                  │
│    - Incluye: Parques Nacional Thingvellir, Geysir y Gullfoss.                                                  │
│  - **Fuentes Termales**: Parada en Secret Lagoon.                                                               │
│    - **Costo**: 3,000 ISK por persona (6,000 ISK total).                                                        │
│  - **Transporte**: Usar el vehículo alquilado.                                                                  │
│                                                                                                                 │
│  #### **Subtotal Día 2**: 39,000 ISK                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Qué define este patrón

El manager decide en runtime cuánto delega a cada agente. Puede consultar poco a transporte y volver dos veces a actividades si la petición lo requiere.